In [3]:
import os
import cv2
import numpy as np
import json

folder = 'stereo_images'
folder1 = 'stereo_output'
cam_info_file = 'camera_info.json'

# Load camera parameters
with open(cam_info_file, 'r') as f:
    cam_info = json.load(f)

K = cam_info['K']  # [fx, 0, cx, 0, fy, cy, 0, 0, 1]
P = cam_info['P']  # projection matrix

fx = K[0]
cx = K[2]
fy = K[4]
cy = K[5]
Tx = 0.2  # baseline (in meters)

print(f"Using fx={fx}, fy={fy}, cx={cx}, cy={cy}, baseline={Tx} m")

# SGBM setup
min_disp = -2
num_disp = 16 * 4
block_size = 5
sgbm = cv2.StereoSGBM_create(
    minDisparity=min_disp,
    numDisparities=num_disp,
    blockSize=block_size,
    P1=8 * 3 * block_size ** 2,
    P2=32 * 3 * block_size ** 2,
    disp12MaxDiff=1,
    uniquenessRatio=10,
    speckleWindowSize=100,
    speckleRange=32
)

# Loop through saved image pairs
i = 0
while True:
    left_path = os.path.join(folder, f'left_{i}.png')
    right_path = os.path.join(folder, f'right_{i}.png')
    if not os.path.exists(left_path) or not os.path.exists(right_path):
        break

    left = cv2.imread(left_path, cv2.IMREAD_GRAYSCALE)
    right = cv2.imread(right_path, cv2.IMREAD_GRAYSCALE)

    disp = sgbm.compute(left, right).astype(np.float32) / 16.0
    disp_clean = cv2.medianBlur(disp, 5)

    depth = np.zeros_like(disp_clean, dtype=np.float32)
    valid = disp_clean > 0
    depth[valid] = (fx * Tx) / disp_clean[valid]

    # Save output
    cv2.imwrite(os.path.join(folder1, f'disparity_{i}.png'), (disp_clean * 16).astype(np.uint16))
    cv2.imwrite(os.path.join(folder1, f'depth_{i}.png'), (depth * 1000).astype(np.uint16))  # in mm

    print(f"Processed pair #{i}")
    i += 1


Using fx=864.9920654296875, fy=864.9921798706055, cx=640.0, cy=480.0, baseline=0.2 m
Processed pair #0
Processed pair #1
Processed pair #2
Processed pair #3
Processed pair #4
Processed pair #5
Processed pair #6
Processed pair #7
Processed pair #8
Processed pair #9
Processed pair #10
Processed pair #11
Processed pair #12
Processed pair #13
Processed pair #14
Processed pair #15
Processed pair #16
Processed pair #17
Processed pair #18
Processed pair #19
Processed pair #20
Processed pair #21
Processed pair #22
Processed pair #23
Processed pair #24
Processed pair #25
Processed pair #26
Processed pair #27
Processed pair #28
Processed pair #29
Processed pair #30
Processed pair #31
Processed pair #32
Processed pair #33
Processed pair #34
Processed pair #35
Processed pair #36
Processed pair #37
Processed pair #38
Processed pair #39
Processed pair #40
Processed pair #41
Processed pair #42
Processed pair #43
Processed pair #44
Processed pair #45
Processed pair #46
Processed pair #47
Processed pai

KeyboardInterrupt: 

In [1]:
#!/usr/bin/env python3
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import CameraInfo
import json
import os

class CameraInfoSaver(Node):
    def __init__(self):
        super().__init__('camera_info_saver')

        self.save_path = 'camera_info.json'  # output file
        self.subscription = self.create_subscription(
            CameraInfo,
            '/camera/front_right/camera_info',  # update topic name if different
            self.callback,
            10
        )
        self.received = False

    def callback(self, msg):
        if self.received:
            return

        cam_info = {
            'width': msg.width,
            'height': msg.height,
            'distortion_model': msg.distortion_model,
            'K': list(msg.k),  # 3x3 camera matrix
            'D': list(msg.d),  # distortion coefficients
            'R': list(msg.r),  # rectification matrix
            'P': list(msg.p),  # projection matrix
        }

        # Save to JSON
        with open(self.save_path, 'w') as f:
            json.dump(cam_info, f, indent=4)

        self.get_logger().info(f"Camera info saved to {self.save_path}")
        self.received = True
        rclpy.shutdown()

def main(args=None):
    rclpy.init(args=args)
    node = CameraInfoSaver()
    rclpy.spin(node)

if __name__ == '__main__':
    main()


[INFO] [1749388824.900469162] [camera_info_saver]: Camera info saved to camera_info.json


KeyboardInterrupt: 

In [6]:
import os
import cv2
import numpy as np
import json

# ====== Config ======
folder = 'stereo_images'
folder_out = 'stereo_output_variants'
cam_info_file = 'camera_info.json'

left_img = os.path.join(folder, 'left_404.png')
right_img = os.path.join(folder, 'right_404.png')
baseline = 0.2  # in meters

# ====== Load Camera Info ======
with open(cam_info_file, 'r') as f:
    cam_info = json.load(f)
K = cam_info['K']
fx = K[0]

# ====== Load Images ======
left = cv2.imread(left_img, cv2.IMREAD_GRAYSCALE)
right = cv2.imread(right_img, cv2.IMREAD_GRAYSCALE)

# ====== SGBM Variants ======
sgbm_settings = [
    {"num_disp": 16*2, "block_size": 3, "uniq": 5},
    {"num_disp": 16*3, "block_size": 5, "uniq": 10},
    {"num_disp": 16*4, "block_size": 7, "uniq": 15},
    {"num_disp": 16*2, "block_size": 9, "uniq": 10},
    {"num_disp": 16*3, "block_size": 11, "uniq": 20},
    {"num_disp": 16*5, "block_size": 3, "uniq": 10},
    {"num_disp": 16*6, "block_size": 5, "uniq": 15},
    {"num_disp": 16*4, "block_size": 7, "uniq": 20},
    {"num_disp": 16*2, "block_size": 11, "uniq": 30},
    {"num_disp": 16*8, "block_size": 3, "uniq": 25},
    {"num_disp": 16*6, "block_size": 9, "uniq": 15},
    {"num_disp": 16*10, "block_size": 5, "uniq": 10},
]

os.makedirs(folder_out, exist_ok=True)

for idx, cfg in enumerate(sgbm_settings):
    sgbm = cv2.StereoSGBM_create(
        minDisparity=0,
        numDisparities=cfg["num_disp"],
        blockSize=cfg["block_size"],
        P1=8 * 3 * cfg["block_size"] ** 2,
        P2=32 * 3 * cfg["block_size"] ** 2,
        disp12MaxDiff=1,
        uniquenessRatio=cfg["uniq"],
        speckleWindowSize=100,
        speckleRange=32
    )

    disp = sgbm.compute(left, right).astype(np.float32) / 16.0
    disp_clean = cv2.medianBlur(disp, 5)

    depth = np.zeros_like(disp_clean, dtype=np.float32)
    valid = disp_clean > 0
    depth[valid] = (fx * baseline) / disp_clean[valid]

    disp_path = os.path.join(folder_out, f'disparity_cfg{idx}.png')
    depth_path = os.path.join(folder_out, f'depth_cfg{idx}.png')

    cv2.imwrite(disp_path, (disp_clean * 16).astype(np.uint16))
    cv2.imwrite(depth_path, (depth * 1000).astype(np.uint16))  # mm

    print(f"Config #{idx} done → block={cfg['block_size']} disp={cfg['num_disp']} uniq={cfg['uniq']}")


Config #0 done → block=3 disp=32 uniq=5
Config #1 done → block=5 disp=48 uniq=10
Config #2 done → block=7 disp=64 uniq=15
Config #3 done → block=9 disp=32 uniq=10
Config #4 done → block=11 disp=48 uniq=20
Config #5 done → block=3 disp=80 uniq=10
Config #6 done → block=5 disp=96 uniq=15
Config #7 done → block=7 disp=64 uniq=20
Config #8 done → block=11 disp=32 uniq=30
Config #9 done → block=3 disp=128 uniq=25
Config #10 done → block=9 disp=96 uniq=15
Config #11 done → block=5 disp=160 uniq=10
